In [1]:
import cobra
import pandas as pd 

from Bio.Seq import Seq
from Bio.Alphabet import generic_rna

import sys
sys.path.insert(1, '../scripts/') # comment out in python script
from load_environmental_variables import *
import gene_information as gi

load environmental variables
The project root is: /Users/joycebaghdassarian/Documents/UCSD/Lewis_Lab/Projects/human_me/


In [4]:
psim_me = pd.read_csv(local_data_path + 'processed/psim_me.csv', index_col = 0)
human_model = cobra.io.load_json_model(local_data_path + 'processed/corrected_recon2_2.json')
sp_dict = {1: True, 0: False}
ptm_cols = ['DSB', 'GPI', 'NG', 'OG']
ptm_keys = list(gi.allowed_ptms.keys())

# gene catalyzing metabolic reaction, not processed via secretory pathway
gene1_id = human_model.genes[1].id
# get information from the ME psim that was built (can do user provided information instead)
idx  = psim_me[psim_me['HGNC ID'] == gene1_id].index
ptms_ = dict(zip(ptm_keys, psim_me.loc[idx, ptm_cols].iloc[0,:].tolist()))
ptms_ = {k:v for k,v in ptms_.items() if v != 0}
ptms_['Phosphorylation'] = 3 # example of one that will not be considered
psim_me.Location = psim_me.Location.replace(float('nan'), '0')
fl = [i.replace('[', '') for i in psim_me.loc[idx, 'Location'].tolist()]
fl = [i.replace(']', '') for i in fl]
ensg_ = psim_me.loc[idx, 'Ensembl gene ID'].tolist()[0]

# initialize the gene class
gene1 = gi.gene_information(human_model, hgnc_id = gene1_id, ptms = ptms_, 
                         tmd = psim_me.loc[idx,'TMD'].tolist()[0], sp = sp_dict[psim_me.loc[idx,'SP'].tolist()[0]], 
                           keff = None)
gene1.get_final_locations(metabolic_model = human_model, final_locations=fl)
gene1.get_sequences(ensg_id = ensg_)
gene1.check_gene_information()
print(gene1.module)
print(gene1.hgnc_id)
print(gene1.sp)
print(gene1.ptms)
print(gene1.tmd)
print(gene1.final_locations)


# start of actual script

In [514]:
human_model = cobra.io.load_json_model(local_data_path + 'processed/corrected_recon2_2.json')
seq_metabolite_map = {human_model.metabolites.get_by_id('utp[n]'): 'U' , 
                      human_model.metabolites.get_by_id('gtp[n]'): 'G',
                      human_model.metabolites.get_by_id('ctp[n]'): 'C',
                      human_model.metabolites.get_by_id('atp[n]'): 'A'}

seq_element_map = dict()
for k,v in seq_metabolite_map.items():
    elements = k.elements
    elements['O'] = elements['O'] - 7
    elements['P'] = elements['P'] - 2
    elements['H'] = elements['H'] - 1 
    seq_element_map[v] = elements
ppi_n = human_model.metabolites.get_by_id('ppi[n]')

In [515]:
class Transcript():
    def __init__(self, gene_information):
        '''Input is an object of the gene_information class, output is all the information needed to 
        build the transcription reations.'''
        self.premrna_seq = Seq(gene_information.premrna_seq, generic_rna)
        self.mrna_seq = Seq(gene_information.mrna_seq, generic_rna)
        self.id = gene_information.hgnc_id
        self.premrna_base_counts = dict()
        for base_letter in seq_element_map.keys():
            self.premrna_base_counts[base_letter] = self.premrna_seq.count(base_letter)
        
        # metabolite output of transcriptional elongation reaction------------------------------------
        self.elongated_transcript = cobra.Metabolite(self.id + '_elongated_transcript[n]')
        self.elongated_transcript.compartment = 'n'
        
        elements = {'C': 0, 'H': 0, 'N': 0, 'O': 0, 'P': 0}
        for base_letter in seq_element_map.keys():
            for element in elements.keys():
                elements[element] += self.premrna_base_counts[base_letter]*seq_element_map[base_letter][element]
        
        #3 and 5' ends
        elements['P'] += 2
        elements['O'] += 7
        elements['H'] += 1
        self.elongated_transcript.elements = elements
        self.elongated_transcript.charge = -len(self.premrna_seq) - 3 # -3 for 5' end triphosphate
        
        # capping is ignored for now------------------------------------
        
        # metabolite output of ------------------------------------
    def transcript_elongation_reaction(self):
        '''Input is an object of the Transcript class. Output is a reaction (cobra.Reaction object) for
        transcriptional elongation of that gene.'''

        # https://www.google.com/search?q=rna+polymerization+reaction&source=lnms&tbm=isch&sa=X&ved=2ahUKEwiN_73Vk7rqAhXOsJ4KHW5lB4UQ_AUoAXoECA4QAw&biw=1920&bih=1001#imgrc=w7XH4mHmJglCuM
        transcript_elongation = cobra.Reaction(self.id + '_transcription_elongation')
        transcript_elongation.subsytem = 'Transcription'
        rxn = dict()
        for ntp, base_letter in seq_metabolite_map.items():
            rxn[ntp] = -1*self.premrna_base_counts[base_letter]
        rxn[ppi_n] = len(self.premrna_seq) - 1 # pyrophosphate released per base added, -1 for 3/5' ends
        rxn[self.elongated_transcript] = 1
        
        # ATP consumption due to PTMs of nucleosomes
        # https://www.pnas.org/content/pnas/suppl/2015/10/29/1514974112.DCSupplemental/pnas.1514974112.sapp.pdf
       # can perhaps add later

        
        transcript_elongation.add_metabolites(rxn)
        # to do: GPRs

        self.transcript_elongation = transcript_elongation  
        
    def transcript_processing(self):
        '''Input is an obj'''
        # https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5027499/

In [516]:
self = Transcript(gene1)

In [517]:
self.transcript_elongation_reaction()

In [525]:
self.transcript_elongation.reaction

'9997 atp[n] + 9143 ctp[n] + 8485 gtp[n] + 9253 utp[n] --> HGNC:80_elongated_transcript[n] + 36877 ppi[n]'

In [512]:
# do gprs then move on

Seq('GACUGGGAAGACAUGACCUUACACACCUUGGUUCUUUAACUUGACCUUGGGAAG...AUU', RNAAlphabet())

In [513]:
self.mrna_seq

Seq('GACUGGGAAGACAUGACCUUACACACCUUGGUUCUUUAACUUGACCUUGGGAAG...AUU', RNAAlphabet())

In [485]:
# https://www.google.com/search?q=rna+polymerization+reaction&source=lnms&tbm=isch&sa=X&ved=2ahUKEwiN_73Vk7rqAhXOsJ4KHW5lB4UQ_AUoAXoECA4QAw&biw=1920&bih=1001#imgrc=w7XH4mHmJglCuM
transcript_elongation = cobra.Reaction(gene_transcript.id + '_transcription_elongation')
transcript_elongation.subsytem = 'Transcription'
rxn = dict()
for ntp, base_letter in seq_metabolite_map.items():
    rxn[ntp] = -1*gene_transcript.premrna_base_counts[base_letter]
rxn[ppi_n] = len(gene_transcript.premrna_seq) - 1 # pyrophosphate released per base added
rxn[gene_transcript.elongated_transcript] = 1
transcript_elongation.add_metabolites(rxn)


In [487]:
transcript_elongation

Reaction identifier,HGNC:80_transcription_elongation
Name,
Memory address,0x07fe6f58d4828
Stoichiometry,9997 atp[n] + 9143 ctp[n] + 8485 gtp[n] + 9253 utp[n] --> HGNC:80_elongated_transcript[n] + 36877 ppi[n] 9997 ATP(4-) + 9143 CTP(4-) + 8485 GTP(4-) + 9253 UTP(4-) --> + 36877 diphosphate(3-)
GPR,
Lower bound,0.0
Upper bound,1000.0


In [475]:
gene_transcript.premrna_seq[0]

'G'